# Loss and Generation Demo

Builds a tiny (untrained, randomly initialized) model, checks that the
next-token cross-entropy loss starts near `log(vocab_size)` as expected for
a random model, then demonstrates autoregressive generation with the KV
cache. For an actually coherent model, run `python pretrain.py --config
configs/gpt_nano.yaml` first and load that checkpoint instead.

In [ ]:
import sys
sys.path.insert(0, '..')
import math
import torch

from src.model.config import GPTConfig
from src.model.transformer import GPT
from src.tokenizer.bpe import BPETokenizer
from src.inference.generator import generate

with open('../data/raw/corpus.txt') as f:
    text = f.read()

tok = BPETokenizer.train(text, vocab_size=512, verbose=False)
print(f'tokenizer vocab size: {tok.vocab_size}')

In [ ]:
cfg = GPTConfig(vocab_size=tok.vocab_size, block_size=64, d_model=128,
                 n_layers=4, n_heads=4, n_kv_heads=4, d_ff=256)
model = GPT(cfg)
print(f'model has {model.num_params():,} non-embedding parameters')

## Sanity check: untrained loss should be close to log(vocab_size)

A freshly initialized model with no training signal should predict
(roughly) a uniform distribution over the vocabulary, so its cross-entropy
loss should be close to `ln(vocab_size)` -- if it's wildly different, that
usually points to an initialization or loss-computation bug.

In [ ]:
ids = tok.encode(text)[:cfg.block_size * 4]
tokens = torch.tensor(ids).view(4, cfg.block_size)

with torch.no_grad():
    _, loss, _ = model(tokens, targets=tokens)

expected = math.log(cfg.vocab_size)
print(f'model loss:            {loss.item():.4f}')
print(f'expected (~uniform):   {expected:.4f}')

## Generate from the (untrained) model

Expect gibberish here -- the point is to exercise the generation +
KV-cache + sampling code path end to end, not to produce good text.
Train the model first (see `pretrain.py`) for coherent output.

In [ ]:
out = generate(model, tok, prompt='Once there was', max_new_tokens=40,
               temperature=0.9, top_k=40)
print(out)